## Quickstart
Concise examples for code construction, syndrome-extraction IR, stim generation, and scheduling.

### 1) CSS Codes

In [ ]:
from macroscheduler.qecc.css import CSS_Code

# Instantiate from parity-check matrices
Hx = [
    [1, 1, 0, 0],
    [0, 1, 1, 0],
]
Hz = [
    [0, 1, 1, 1],
    [1, 0, 0, 1],
]
css_from_h = CSS_Code(Hx=Hx, Hz=Hz)

# Instantiate from explicit X/Z check supports
x_checks = [[0, 1], [1, 2]]
z_checks = [[1, 2, 3], [0, 3]]
css_from_checks = CSS_Code(x_checks=x_checks, z_checks=z_checks, num_data_q=4)

print("[CSS from H] n, rx, rz =", css_from_h.num_data_q, css_from_h.num_x_check, css_from_h.num_z_check)
print("[CSS from checks] n, rx, rz =", css_from_checks.num_data_q, css_from_checks.num_x_check, css_from_checks.num_z_check)

### 2) BB Codes

In [ ]:
from macroscheduler.qecc.qalp import BB_Code

# BB code over Z_l x Z_m with two polynomials A, B
bb = BB_Code(
    l=3,
    m=3,
    A_poly=[[0, 0], [1, 0]],
    B_poly=[[0, 0], [0, 1]],
)
print("BB code n, rx, rz =", bb.num_data_q, bb.num_x_check, bb.num_z_check)

### 3) TT Codes

In [ ]:
from macroscheduler.qecc.qalp import TT_Code

# TT code with three polynomials A, B, C
tt = TT_Code(
    l=2,
    m=2,
    n=3,
    A_poly=[[0, 1, 1], [1, 0, 0]],
    B_poly=[[0, 0, 2], [0, 1, 0]],
    C_poly=[[1, 1, 1], [0, 0, 1]],
)
print("TT code n, rx, rz =", tt.num_data_q, tt.num_x_check, tt.num_z_check)

### 4) Quasi-cyclic LP codes

In [ ]:
from macroscheduler.qecc.qalp import QALP, LiftedMatrix, Perm
from macroscheduler.scheduling import all_schedules

l = 30
A_powers = [
    [0, 0, 0, 0, 0],
    [0, 2, 14, 24, 25],
    [0, 16, 11, 14, 13]
]

m = len(A_powers)
n = len(A_powers[0])

A_tilde = [[[Perm.right_shift(l) ** A_powers[i][j]] for j in range(n)] for i in range(m)]

B_tilde = [[[Perm.right_shift(l) ** (l - A_powers[i][j])] for j in range(n)] for i in range(m)]

A = LiftedMatrix(A_tilde, l)
B = LiftedMatrix(B_tilde, l)

code = QALP([A, B])
print(f"Code has {code.num_data_q} data qubits")

### 5) Stim Generation + Scheduling IR
The SE schedule IR is `(s_x_check, s_z_check, depth)` and is the bridge between scheduling (`macroscheduler.scheduling`) and circuit synthesis (`macroscheduler.qecc.stim`).

In [ ]:
import stim
from macroscheduler.qecc.stim import generate_stim_str
from macroscheduler.scheduling import random_schedule

# Sample one valid schedule and load it into the code's IR
s_x, s_z = random_schedule(code)
code.set_se_schedule_ir(s_x, s_z)

p_2q_projection = 0.01 * 4 / 15
p_1q_projection = 0.001 * 2 / 3

stim_str = generate_stim_str(
    code, rounds=12, obs_type='Z', 
    p_2q_channel=f'PAULI_CHANNEL_2({p_2q_projection},0,0,{p_2q_projection},{p_2q_projection},0,0,0,0,0,0,0,0,0,0)', 
    p_idle_channel=None,
    p_meas=0.001
)
stim.Circuit(stim_str).detector_error_model(approximate_disjoint_errors=True)
print(stim_str.splitlines()[:8])

dem = stim.Circuit(stim_str).detector_error_model(approximate_disjoint_errors=True)
print("Detector error model terms:", len(str(dem).splitlines()))

### 7) `macroscheduler.scheduling`: all schedules + random schedule

In [ ]:
from macroscheduler.scheduling import all_schedules, random_schedule

# Enumerate all schedules for the [[56,6,6]] instance
all_scheds = all_schedules(code)
print("Number of schedules:", len(all_scheds))

# Draw one random schedule from the same space
s_x_rand, s_z_rand = random_schedule(code)
code.set_se_schedule_ir(s_x_rand, s_z_rand)
print("Random schedule depth:", code.depth)

In [ ]:
from macroscheduler.scheduling import all_schedules, random_schedule

# Enumerate all schedules for the [[56,6,6]] instance
all_scheds = all_schedules(code)
print("Number of schedules:", len(all_scheds))